In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

In [ ]:
# prompt: read in a .csv file called flare_well_sound_pollutant_met_data.csv.csv, then show the first few rows of the data

import pandas as pd

# Assuming the file is in the current working directory.
# If not, provide the full path to the file.
try:
    df = pd.read_csv('flare_well_sound_pollutant_met_data.csv')
    print(df.head())
except FileNotFoundError:
    print("Error: 'flare_well_sound_pollutant_met_data.csv' not found. Please make sure the file exists in the current directory or provide the correct path.")



In [ ]:
# Renaming columns by replacing spaces with underscores
df.rename(columns=lambda col: col.replace(" ", "_"), inplace=True)

# Output the updated column names
print(df.columns)

In [ ]:
# Output the updated column names
# Print all column names in a readable format
print(list(df.columns))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fftpack import fft, fftfreq

# Remove missing values
df_clean = df.dropna(subset=["LZeq_12.5Hz", "time_utc"])

# Convert "time_utc" from string to datetime (handling mixed formats)
df_clean["time_utc"] = pd.to_datetime(df_clean["time_utc"], errors="coerce")

# Drop any rows that failed conversion
df_clean = df_clean.dropna(subset=["time_utc"])

# Convert to seconds relative to the first timestamp
df_clean["time_seconds"] = (df_clean["time_utc"] - df_clean["time_utc"].min()).dt.total_seconds()

# Extract time and signal values
time = df_clean["time_seconds"].values
signal = df_clean["LZeq_12.5Hz"].values

# Compute sampling rate
dt = np.mean(np.diff(time))  # Average time difference
fs = 1 / dt  # Sampling frequency (Hz)

# Perform FFT
N = len(signal)  # Number of data points
fft_values = fft(signal)  # Compute FFT
frequencies = fftfreq(N, d=dt)  # Compute frequency bins

# Keep only positive frequencies
positive_freqs = frequencies[:N // 2]
positive_fft_values = np.abs(fft_values[:N // 2])

# Plot the frequency spectrum
plt.figure(figsize=(10, 5))
plt.plot(positive_freqs, positive_fft_values)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title("Frequency Spectrum of LZeq_12.5Hz")
plt.grid()
plt.show()


In [ ]:
# Define predictors and response
predictors = [
    "LAFmax", "LAFmin", "LCpeak", "LAeq", "LCeq", "LZeq_12.5Hz",
    "LZeq_16Hz", "LZeq_20Hz", "LZeq_25Hz", "LZeq_31.5Hz", "LZeq_40Hz",
    "LZeq_50Hz", "LZeq_63Hz", "LZeq_80Hz", "LZeq_100Hz", "LZeq_125Hz",
    "LZeq_160Hz", "LZeq_200Hz", "LZeq_250Hz", "LZeq_315Hz", "LZeq_400Hz",
    "LZeq_500Hz", "LZeq_630Hz", "LZeq_800Hz", "LZeq_1kHz", "LZeq_1.25kHz",
    "LZeq_1.6kHz", "LZeq_2kHz", "LZeq_2.5kHz", "LZeq_3.15kHz", "LZeq_4kHz",
    "LZeq_5kHz", "LZeq_6.3kHz", "LZeq_8kHz", "LZeq_10kHz", "LZeq_12.5kHz",
    "LZeq_16kHz", "num_oil_wells", "num_gas_wells", "avg_monthly_oil",
    "avg_monthly_gas", "temp_f", "wsp_ms", "wdr_deg", "relh_percent",
    "pressure_altcorr", "water_vapor_mr_g_kg", "solr_W_m2", "temp_bb"
]
response = "so2"

# Step 1: Check and convert data to float
for col in predictors + [response]:
    if not np.issubdtype(df[col].dtype, np.floating):
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing values after conversion
df = df.dropna(subset=predictors + [response])

# Step 2: Split the data
train_data, temp_data = train_test_split(df, test_size=0.4, random_state=42)
valid_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)



In [ ]:
print(df.columns)


# Linear Regression: All Predictors

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Step 3: Define predictors and response for training and validation
X_train = train_data[predictors]
y_train = train_data[response]

X_valid = valid_data[predictors]
y_valid = valid_data[response]

# Step 4: Fit Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 5: Make predictions on validation data
y_pred = model.predict(X_valid)

# Step 6: Calculate R-squared
r2 = r2_score(y_valid, y_pred)

# Print R-squared value
print("R-squared:", r2)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Step 7: Test data evaluation
X_test = test_data[predictors]
y_test = test_data[response]

# Make predictions on test data
y_test_pred = model.predict(X_test)

# Calculate R-squared for test data
test_r2 = r2_score(y_test, y_test_pred)

# Calculate Mean Absolute Error (MAE)
test_mae = mean_absolute_error(y_test, y_test_pred)

# Calculate Root Mean Squared Error (RMSE)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

# Print evaluation metrics
print("Test R-squared:", test_r2)
print("Test MAE:", test_mae)
print("Test RMSE:", test_rmse)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate residuals
residuals = y_valid - y_pred

# Residuals vs Predicted Values Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_pred, y=residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values")
plt.grid(True)
plt.show()

# Histogram of Residuals Plot
plt.figure(figsize=(8, 6))
sns.histplot(residuals, kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Histogram of Residuals")
plt.grid(True)
plt.show()


# XGBoost: All Predictors

In [ ]:
# Prepare data for XGBoost
dtrain = xgb.DMatrix(train_data[predictors], label=train_data[response])
dvalid = xgb.DMatrix(valid_data[predictors], label=valid_data[response])
dtest = xgb.DMatrix(test_data[predictors], label=test_data[response])

In [ ]:
best_params = {'objective': 'reg:squarederror', 'max_depth': 12, 'eta': 0.01, 'subsample': 0.65, 'colsample_bytree': 1.0, 'eval_metric': 'rmse'}

In [ ]:
xgb_all_model = xgb.train(best_params, dtrain, num_boost_round=1000)

In [ ]:
# Step 5: Evaluate on test data
preds = xgb_all_model.predict(dvalid)

In [ ]:
from sklearn.metrics import r2_score


# Calculate R-squared for training data
valid_r2 = r2_score(dvalid.get_label(), preds)

# Print the R-squared value
print("Training R-squared:", valid_r2)


In [ ]:
# Predictions on test data
preds_test = xgb_all_model.predict(dtest)

# Calculate R-squared for test data
test_r2 = r2_score(dtest.get_label(), preds_test)

# Calculate Mean Absolute Error (MAE)
test_mae = mean_absolute_error(dtest.get_label(), preds_test)

# Calculate Root Mean Squared Error (RMSE)
test_rmse = np.sqrt(mean_squared_error(dtest.get_label(), preds_test))

# Print test evaluation metrics
print("Test R-squared:", test_r2)
print("Test MAE:", test_mae)
print("Test RMSE:", test_rmse)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate residuals for validation data
residuals_valid = dvalid.get_label() - preds

# Residuals vs Predicted Values Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x=preds, y=residuals_valid)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values - XGBoost Model")
plt.grid(True)
plt.show()

# Histogram of Residuals Plot
plt.figure(figsize=(8, 6))
sns.histplot(residuals_valid, kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Histogram of Residuals - XGBoost Model")
plt.grid(True)
plt.show()


XGBoost: Selected Predictors

In [ ]:
# Define predictors and response
predictors = [
    "LAFmax", "LAFmin", "LCpeak", "LAeq", "LZeq_12.5Hz",
    "LZeq_25Hz",
    "LZeq_63Hz", "LZeq_80Hz", "LZeq_100Hz", "LZeq_125Hz",
    "LZeq_160Hz", "LZeq_200Hz", "LZeq_250Hz",
    "LZeq_500Hz","LZeq_1.25kHz",
    "LZeq_1.6kHz",
    "LZeq_5kHz","LZeq_12.5kHz",
    "num_oil_wells", "num_gas_wells", "avg_monthly_oil",
    "avg_monthly_gas", "temp_f", "wsp_ms", "wdr_deg", "relh_percent",
    "pressure_altcorr", "water_vapor_mr_g_kg", "solr_W_m2"
]
response = "nox"

# Step 1: Check and convert data to float
for col in predictors + [response]:
    if not np.issubdtype(df[col].dtype, np.floating):
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing values after conversion
select_df = df.dropna(subset=predictors + [response])

# Step 2: Split the data
train_data_select, temp_data_select = train_test_split(select_df, test_size=0.4, random_state=42)
valid_data_select, test_data_select = train_test_split(temp_data_select, test_size=0.5, random_state=42)

In [ ]:
# Prepare data for XGBoost
dtrain_select = xgb.DMatrix(train_data_select[predictors], label=train_data_select[response])
dvalid_select = xgb.DMatrix(valid_data_select[predictors], label=valid_data_select[response])
dtest_select = xgb.DMatrix(test_data_select[predictors], label=test_data_select[response])

In [ ]:
best_params = {'objective': 'reg:squarederror', 'max_depth': 12, 'eta': 0.01, 'subsample': 0.65, 'colsample_bytree': 1.0, 'eval_metric': 'rmse'}

In [ ]:
xgb_select_model = xgb.train(best_params, dtrain_select, num_boost_round=1000)

In [ ]:
# Step 5: Evaluate on test data
preds_select = xgb_select_model.predict(dvalid_select)

In [ ]:
from sklearn.metrics import r2_score


# Calculate R-squared for training data
valid_r2_select = r2_score(dvalid_select.get_label(), preds_select)

# Print the R-squared value
print("Training R-squared:", valid_r2_select)

In [ ]:
# Predictions on test data
preds_test_selected = xgb_select_model.predict(dtest_select)

# Calculate R-squared for test data
test_r2_selected = r2_score(dtest_select.get_label(), preds_test_selected)

# Calculate Mean Absolute Error (MAE)
test_mae_selected = mean_absolute_error(dtest_select.get_label(), preds_test_selected)

# Calculate Root Mean Squared Error (RMSE)
test_rmse_selected = np.sqrt(mean_squared_error(dtest_select.get_label(), preds_test_selected))

# Print test evaluation metrics
print("Test R-squared:", test_r2_selected)
print("Test MAE:", test_mae_selected)
print("Test RMSE:", test_rmse_selected)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate residuals for validation data
residuals_select = dvalid_select.get_label() - preds_select

# Residuals vs Predicted Values Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x=preds_select, y=residuals_select)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values - XGBoost Model (Selected Predictors)")
plt.grid(True)
plt.show()

# Histogram of Residuals Plot
plt.figure(figsize=(8, 6))
sns.histplot(residuals_select, kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Histogram of Residuals - XGBoost Model (Selected Predictors)")
plt.grid(True)
plt.show()


# Variable Importance of final model (XGBoost with Selected Predictors)

In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance with adjusted grid size
plt.figure(figsize=(18, 24))  # Increase size for better spacing
xgb.plot_importance(xgb_select_model, importance_type='weight', max_num_features=20, height=0.5)
plt.title("Variable Importance (Top 20)", fontsize=14)
plt.xlabel("F score", fontsize=12)
plt.ylabel("Features", fontsize=12)
# Extend x-axis limit to fit the largest value
plt.xlim(0, 160000)  # Adjust this value based on your data range
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()  # Automatically adjust subplot parameters to fit the plot
plt.show()